# Histogramas

Histogramas de imagens são primariamente usados como ferramenta de análise em visão computacional uma vez que quantificam a distribuição dos dados associados a imagem (por exemplo, valores de intensidade dentro da imagem). Nesse notebook vamos demonstrar como produzir histogramas das imagens e interpretá-los.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Nesse notebook, vamos usar as mesmas bibliotecas que trabalhamos nos notebooks anteriores.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image

## 1. Introdução aos Histogramas

Histogramas são contagens agrupadas de dados organizadas em um conjunto de intervalos predefinidos, chamados de bins (ou classes/faixas). Quando criamos um gráfico de histograma, precisamos especificar a quantidade de bins ao longo do eixo x.

Cada bin representa uma faixa de valores, como por exemplo: intensidades de pixel de 0 a 9, 10 a 19, 20 a 29 e assim por diante. Em outras palavras, cada bin funciona como um “compartimento” que agrupa valores semelhantes e contabiliza quantas ocorrências existem dentro daquela faixa.

Veremos vários exemplos a seguir que ajudarão a consolidar esse conceito.

---

### <font color="green">Sintaxe da Função</font>
``` python
retval = plt.hist(x[, bins[, range[, ...]]])
```

`retval`: Array (vetor) ou lista de arrays contendo os valores dos bins do histograma. Se a entrada for uma sequência de arrays `[data1, data2, ...]`, então o retorno será uma lista de arrays com os valores dos histogramas correspondentes a cada array, na mesma ordem. O tipo de dado (`dtype`) do array `n` (ou dos arrays que o compõem) será sempre **float**, mesmo quando não houver ponderação (*weighting*) ou normalização.

A função possui **1 argumento de entrada obrigatório** e vários parâmetros opcionais:

1. `x`: Array ou sequência de arrays de dimensão `(n,)`. Contém os valores de entrada. Pode ser um único array ou uma sequência de arrays, que não precisam ter o mesmo tamanho.
2. `bins`: Define a quantidade de bins (intervalos de mesma largura) dentro da faixa de valores considerada. Este é um **argumento opcional** cujo valor padrão é **10**.
3. `range`: Define os limites inferior e superior utilizados para criar os bins. Valores abaixo do limite inferior ou acima do limite superior são ignorados. Este é um **argumento opcional** cujo valor padrão é `None`, o que equivale a utilizar toda a faixa de valores presente na entrada `x`.

## 2. Histograma de Imagens em Escala de Cinza

### 2.1. Exemplo 1

In [ ]:
# Lê a imagem
img = cv2.imread('imagens/MNIST_3_18x18.png', cv2.IMREAD_GRAYSCALE)

# Converte array 2D em um array 1D
img_flatten = img.ravel()

# Apresenta imagem e histogramas em diferentes bins
plt.figure(figsize=[18, 4])

plt.subplot(131); plt.imshow(img, cmap='gray'); plt.title('Original');
plt.subplot(132); plt.hist(img_flatten, 5);  plt.xlim([0, 256]); plt.title('5 Bins');
plt.subplot(133); plt.hist(img_flatten, 50); plt.xlim([0, 256]); plt.title('50 Bins');

### 2.2. Exemplo 2

In [ ]:
img = cv2.imread('imagens/puc-minas.jpg', cv2.IMREAD_GRAYSCALE)

# Converte array 2D em um array 1D
img_flatten = img.ravel()

# Apresenta imagem e histogramas em diferentes bins
plt.figure(figsize=[18, 4])

plt.subplot(131); plt.imshow(img, cmap='gray'); plt.title('Original');
plt.subplot(132); plt.hist(img_flatten, 50);  plt.xlim([0, 256]); plt.title('50 Bins');
plt.subplot(133); plt.hist(img_flatten, 256); plt.xlim([0, 256]); plt.title('256 Bins');

### 2.3. Comparando calcHist() com plt.hist()

---

### <font color="green">Sintaxe da Função</font>
```python
    hist = cv2.calcHist(images, channels, mask, histSize, ranges[, hist[, accumulate]])
```

Esta função possui **5 argumentos obrigatórios**:

1. `images`: Arrays de origem (imagens de entrada). Todas devem possuir a mesma profundidade de dados e o mesmo tamanho. Cada imagem pode ter um número arbitrário de canais.
2. `channels`: Lista dos canais utilizados para calcular o histograma. Os canais da primeira imagem são numerados de **0** até **images\[0].channels() - 1**. Os canais da segunda imagem são numerados a partir de **images\[0].channels()** até **images\[0].channels() + images\[1].channels() - 1**, e assim sucessivamente.
3. `mask`: Máscara opcional. Se não estiver vazia, deve ser um array de **8 bits** com o mesmo tamanho de **images\[i]**. Os elementos da máscara com valor diferente de zero indicam quais elementos da imagem serão considerados no cálculo do histograma.
4. `histSize`: Array que define o número de bins (intervalos) do histograma em cada dimensão.
5. `ranges`: Array contendo, para cada dimensão do histograma, os limites que definem as fronteiras dos bins. Esses limites determinam os intervalos de valores utilizados para agrupar os dados no histograma.

In [ ]:
img = cv2.imread('imagens/puc-minas.jpg', cv2.IMREAD_GRAYSCALE)

# Usando calcHist() do OpenCV.
hist = cv2.calcHist(images=[img], channels=[0], mask=None, histSize=[256], ranges=[0, 255])

# Converte array 2D em um array 1D
img_flatten = img.ravel()

# Apresenta imagem e histogramas
plt.figure(figsize = [18, 4])
plt.subplot(131); plt.imshow(img, cmap='gray'); plt.title('Original Image');
plt.subplot(132); plt.plot(hist); plt.xlim([0, 256]); plt.title('cv2.calcHist()');
plt.subplot(133); plt.hist(img_flatten, 256); plt.xlim([0, 256]); plt.title('np.ravel(), plt.hist()');

## 3. Histogramas em Imagens Coloridas

### 3.1. Compondo Histogramas para cada Canal de Cor

In [ ]:
# Lendo imagem como colorida
img = cv2.imread('imagens/puc-minas.jpg')

# Calculando histogramas para cada canal de cor
hist1 = cv2.calcHist([img], [0], None, [256], [0, 255])
hist2 = cv2.calcHist([img], [1], None, [256], [0, 255])
hist3 = cv2.calcHist([img], [2], None, [256], [0, 255])

# Apresentando imagem e histogramas
plt.figure(figsize=[18, 10])
plt.subplot(221); plt.imshow(img[:, :, ::-1])

plt.subplot(222) 
plt.plot(hist1, 'b'); plt.plot(hist2, 'g'); plt.plot(hist3, 'r') 
plt.xlim([0, 256]);

### 3.2. Usando Máscaras

In [ ]:
# Lendo imagem como colorida
img = cv2.imread('imagens/puc-minas.jpg')

# Criando uma máscara para filtrar a área de interesse da imagem
mask_hist = np.zeros((img.shape[0], img.shape[1]), dtype='uint8')

# Indicando a área de interesse
mask_hist[30:200, 0:270] = 255

# Criando os canais para a máscara
mat = [mask_hist, mask_hist, mask_hist]
mask = cv2.merge(mat, 3)

# Criando a imagem para demonstração da área de interesse
img_roi = cv2.bitwise_and(img, mask)

# Calculando histogramas para cada canal de cor
# Note que estamos indicando uma máscara
hist1 = cv2.calcHist([img], [0], mask_hist, [256], [0, 255])
hist2 = cv2.calcHist([img], [1], mask_hist, [256], [0, 255])
hist3 = cv2.calcHist([img], [2], mask_hist, [256], [0, 255])

# Apresentando imagem e histogramas
plt.figure(figsize=[18, 10])
plt.subplot(221); plt.imshow(img_roi[:, :, ::-1])

plt.subplot(222) 
plt.plot(hist1, 'b'); plt.plot(hist2, 'g'); plt.plot(hist3, 'r') 
plt.xlim([0, 256]);